# C-tool — ASR + Speaker Diarization → JSON

Chạy [vcstack/c-tools](https://github.com/vcstack/c-tools) trên Colab:

```text
Video / Audio → FFmpeg → WhisperX STT → pyannote → JSON
```

Không dịch, TTS, hay render video.

## Trước khi chạy

1. **Runtime → Change runtime type → GPU** (T4).
2. Hugging Face token: https://hf.co/settings/tokens
3. Accept license:
   - https://huggingface.co/pyannote/speaker-diarization-3.1
   - https://huggingface.co/pyannote/segmentation-3.0

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Không thấy GPU. Runtime → Change runtime type → GPU rồi chạy lại.")

## 1. Clone C-tool + cài dependency

In [ ]:
import os, shutil
from pathlib import Path

os.chdir("/content")
ROOT = Path("/content/c-tools")
if ROOT.exists():
    shutil.rmtree(ROOT)

!git clone --depth 1 https://github.com/vcstack/c-tools.git /content/c-tools
!apt-get -y install -qq ffmpeg

%cd /content/c-tools
!pip install -q python-dotenv soundfile
!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124
!pip install -q "ctranslate2<=4.4.0" "transformers<=4.39.3"
!pip install -q git+https://github.com/R3gm/pyannote-audio.git@3.1.1
!pip install -q git+https://github.com/R3gm/whisperX.git@cuda_12_x

print("Repo:", ROOT)
print("Has main.py:", (ROOT / "main.py").exists())

## 2. Hugging Face token

Cần nếu `max_speakers > 1`. Để trống + `max_speakers=1` thì mọi câu gán `SPEAKER_00`.

In [ ]:
# @title Hugging Face token
HF_TOKEN = ""  # @param {type:"string"}
import os
from pathlib import Path
os.environ["HF_TOKEN"] = HF_TOKEN.strip()
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN.strip()
(Path("/content/c-tools") / ".env").write_text(f"HF_TOKEN={HF_TOKEN.strip()}\n", encoding="utf-8")
print("HF_TOKEN set:", bool(HF_TOKEN.strip()))

## 3. Input

Chạy **một** cell: upload file, hoặc mẫu JFK.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

INPUT_DIR = Path("/content/c-tools/input")
INPUT_DIR.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
if not uploaded:
    raise SystemExit("Chưa upload file.")

name = next(iter(uploaded))
dest = INPUT_DIR / Path(name).name
src = Path("/content/c-tools") / Path(name).name
if not src.exists():
    src = Path("/content") / Path(name).name
if src.exists():
    shutil.move(str(src), dest)
else:
    dest.write_bytes(uploaded[name])

INPUT_PATH = str(dest)
print("Input:", INPUT_PATH)

In [ ]:
from pathlib import Path
Path("/content/c-tools/input").mkdir(parents=True, exist_ok=True)
!wget -q -O /content/c-tools/input/jfk.flac https://github.com/openai/whisper/raw/main/tests/jfk.flac
INPUT_PATH = "/content/c-tools/input/jfk.flac"
print("Input:", INPUT_PATH)

## 4. Chạy pipeline C-tool

Transcript giữ ngôn ngữ gốc, không dịch.

In [ ]:
# @title Run ASR + diarization
model = "large-v3"  # @param ["tiny", "base", "small", "medium", "large-v2", "large-v3"]
language = ""  # @param {type:"string"}
min_speakers = 1  # @param {type:"integer"}
max_speakers = 10  # @param {type:"integer"}
batch_size = 8  # @param {type:"integer"}

import os, sys, json
from pathlib import Path

ROOT = Path("/content/c-tools")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if "INPUT_PATH" not in globals():
    raise SystemExit("Chạy cell upload hoặc cell mẫu JFK trước.")

from pipeline.config import PipelineConfig, resolve_device
from pipeline.pipeline import run_pipeline

OUTPUT_PATH = str(ROOT / "output" / "result.json")
cfg = PipelineConfig(
    asr_model=model,
    batch_size=int(batch_size),
    source_language=language.strip() or None,
    min_speakers=int(min_speakers),
    max_speakers=int(max_speakers),
    device=resolve_device("auto"),
    work_dir=str(ROOT / ".cache" / "pipeline"),
    hf_token=os.environ.get("HF_TOKEN") or None,
)

result = run_pipeline(INPUT_PATH, OUTPUT_PATH, cfg)
print(json.dumps(result, ensure_ascii=False, indent=2)[:4000])

## 5. Tải JSON

In [ ]:
import json
from pathlib import Path
from google.colab import files

out = Path("/content/c-tools/output/result.json")
data = json.loads(out.read_text(encoding="utf-8"))
print("file:", data["source"]["filename"])
print("duration:", data["source"]["duration"], "s")
print("speakers:", data["speakers"])
print("segments:", len(data["segments"]))
print()
for seg in data["segments"][:20]:
    print(f'{seg["start"]:7.2f} → {seg["end"]:7.2f}  {seg["speaker"]}  {seg["text"]}')

files.download(str(out))